In [1]:
import os
from src.datasets.loaders import (
    MetaICLDataset,
    BaselineDataset,
    make_collator,
    read_json,
    filter_records_by_task,
    split_holdout_task,
)
from omegaconf import DictConfig

In [9]:
BASE_DIR = "data/"
ood_task = "CycleCheck"
pool_records = []

cfg = {
    "dataset": {
        "dataset_config": "baseline",
        "ood_task": ood_task,
    }
}

# make cfg object
cfg = DictConfig(cfg)

In [10]:

for split in ["train", "val", "test"]:
    pool_path = os.path.join(BASE_DIR, "graphqa",
                                f"{cfg.dataset.dataset_config}",
                                "ood_pool", f"{split}.json")
    if not os.path.exists(pool_path):
        raise FileNotFoundError(
            f"OOD pool file not found: {pool_path}. "
            f"Generate it by running dataset_generator.py with OOD_POOL_MODE=True."
        )
    pool_records.extend(read_json(pool_path))

available_tasks = sorted({r["task"] for r in pool_records})
if ood_task not in available_tasks:
    raise ValueError(
        f"ood_task '{ood_task}' not found in pool. Available tasks: {available_tasks}"
    )

train_records = filter_records_by_task(pool_records, exclude_task=ood_task)
val_records, test_records = split_holdout_task(pool_records, ood_task)
print(f"[LOTO] held-out task : {ood_task}")
print(f"[LOTO] train tasks   : {sorted({r['task'] for r in train_records})}")
print(f"[LOTO] sample counts : train={len(train_records)} "
        f"val={len(val_records)} test={len(test_records)}")

if cfg.dataset.dataset_config == "meta-icl":
    train_dataset = MetaICLDataset._from_records(train_records, k=cfg.dataset.n_examples)
    val_dataset   = MetaICLDataset._from_records(val_records, k=cfg.dataset.n_examples)
elif cfg.dataset.dataset_config == "baseline":
    train_dataset = BaselineDataset._from_records(train_records)
    val_dataset   = BaselineDataset._from_records(val_records)

[LOTO] held-out task : CycleCheck
[LOTO] train tasks   : ['ConnectedNodes', 'DisconnectedNodes', 'EdgeCount', 'EdgeExistence', 'MaximumFlow', 'NodeCount', 'NodeDegree', 'Reachability', 'ShortestPath', 'TriangleCounting']
[LOTO] sample counts : train=3500 val=175 test=175


In [13]:
train_dataset[0]

{'answer': 'Yes.',
 'prompt': '#### Question\nIn an undirected graph, (i,j) means that node i and node j are connected with an undirected edge. G describes a graph among nodes 0, 1, 2, 3, 4, 5, 6, 7, and 8.\nThe edges in G are: (0, 1) (1, 2) (2, 3) (3, 4) (4, 5) (5, 6) (6, 7) (7, 8).\nQ: Is there a path from node 0 to node 3?\nA: Yes.',
 'algorithm': 'path',
 'task': 'Reachability'}

In [15]:
val_dataset[15]

{'answer': 'Yes, there is a cycle.',
 'prompt': '#### Question\nIn an undirected graph, (i,j) means that node i and node j are connected with an undirected edge. G describes a graph among nodes 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, and 17.\nThe edges in G are: (0, 1) (0, 4) (0, 5) (0, 6) (0, 7) (0, 8) (0, 10) (0, 11) (0, 12) (0, 15) (0, 16) (0, 17) (1, 2) (1, 3) (1, 4) (1, 8) (1, 10) (1, 11) (1, 12) (2, 3) (2, 4) (2, 7) (2, 8) (2, 10) (2, 11) (2, 14) (3, 4) (3, 5) (3, 6) (3, 7) (3, 8) (3, 9) (3, 12) (3, 13) (3, 14) (3, 16) (3, 17) (4, 5) (4, 7) (4, 8) (4, 9) (4, 11) (4, 12) (4, 14) (5, 6) (5, 8) (5, 10) (5, 11) (5, 12) (5, 13) (5, 14) (5, 16) (5, 17) (6, 9) (6, 10) (6, 11) (6, 14) (6, 15) (6, 16) (7, 9) (7, 10) (7, 11) (7, 13) (8, 9) (8, 10) (8, 11) (8, 12) (8, 13) (8, 14) (8, 15) (8, 16) (8, 17) (9, 10) (9, 11) (9, 12) (9, 13) (9, 14) (9, 15) (9, 17) (10, 11) (11, 14) (11, 15) (11, 16) (11, 17) (15, 17).\nQ: Is there a cycle in this graph?\nA: Yes, there is a cycle